In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import csv
import json
from datetime import datetime, date

LIMITE_SUSPEITO = 10000.00
CAMINHO_CSV = "/content/drive/MyDrive/Colab Notebooks/transacoes.csv"
CAMINHO_JSON = "relatorio.json"

In [4]:
def ler_transacoes(caminho_arquivo):
    """
    Lê o arquivo CSV e retorna uma lista com as transações brutas.
    Cada transação será um dicionário.
    """
    transacoes = []

    try:
        with open(caminho_arquivo, "r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                transacoes.append(linha)

    except FileNotFoundError:
        print("Erro: arquivo CSV não encontrado.")
        return []

    return transacoes

In [9]:
def validar_transacao(linha):

    # Validação do ID
    id_texto = linha.get("id", "").strip()

    if not id_texto.isdigit():
        return None

    id_transacao = int(id_texto)

    # Validação do cliente
    cliente_id = linha.get("cliente_id", "").strip()

    if cliente_id == "":
        return None

    # Validação da data
    data_texto = linha.get("data", "").strip()

    try:
        data_transacao = datetime.strptime(data_texto, "%Y-%m-%d")
    except ValueError:
        return None

    # Validação do tipo
    tipo = linha.get("tipo", "").strip().lower()

    if tipo not in ["credito", "debito"]:
        return None

    # Validação do valor
    valor_texto = linha.get("valor", "").strip()

    try:
        valor = float(valor_texto)
    except ValueError:
        return None

    if valor <= 0:
        return None

    # Campos opcionais/textuais
    descricao = linha.get("descricao", "").strip()
    categoria = linha.get("categoria", "").strip()

    transacao_limpa = {
        "id": id_transacao,
        "data": data_transacao,
        "data_texto": data_transacao.strftime("%Y-%m-%d"),
        "mes": data_transacao.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": descricao,
        "categoria": categoria,
        "suspeita": valor > LIMITE_SUSPEITO
    }

    return transacao_limpa

In [11]:
transacoes_brutas = ler_transacoes(CAMINHO_CSV)

transacoes_validas = []
transacoes_invalidas = []

for linha in transacoes_brutas:
    transacao = validar_transacao(linha)

    if transacao is None:
        transacoes_invalidas.append(linha)
    else:
        transacoes_validas.append(transacao)

print("Total de linhas lidas:", len(transacoes_brutas))
print("Linhas válidas:", len(transacoes_validas))
print("Linhas inválidas:", len(transacoes_invalidas))

Total de linhas lidas: 200
Linhas válidas: 185
Linhas inválidas: 15


In [12]:
def gerar_relatorio(transacoes):
    """
    Agrupa as transações válidas por mês e calcula as métricas financeiras.
    """

    resumo_mensal = {}

    for transacao in transacoes:
        mes = transacao["mes"]
        valor = transacao["valor"]
        tipo = transacao["tipo"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": valor,
                "menor_valor": valor,
                "valores": []
            }

        resumo_mensal[mes]["quantidade"] += 1
        resumo_mensal[mes]["valores"].append(valor)

        if tipo == "credito":
            resumo_mensal[mes]["total_credito"] += valor
        elif tipo == "debito":
            resumo_mensal[mes]["total_debito"] += valor

        if valor > resumo_mensal[mes]["maior_valor"]:
            resumo_mensal[mes]["maior_valor"] = valor

        if valor < resumo_mensal[mes]["menor_valor"]:
            resumo_mensal[mes]["menor_valor"] = valor

    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = sum(dados["valores"]) / dados["quantidade"]

        # Removendo a lista auxiliar para deixar o relatório mais limpo
        del dados["valores"]

    return resumo_mensal

In [13]:
resumo_mensal = gerar_relatorio(transacoes_validas)

print("Meses encontrados:")
print(resumo_mensal.keys())

print("\nResumo de um mês:")
print(resumo_mensal["2026-01"])

Meses encontrados:
dict_keys(['2026-06', '2026-02', '2026-05', '2026-03', '2026-04', '2026-01'])

Resumo de um mês:
{'quantidade': 33, 'total_credito': 78195.29, 'total_debito': 20142.780000000002, 'saldo': 58052.509999999995, 'media': 2979.941515151515, 'maior_valor': 12286.58, 'menor_valor': 129.34}


In [14]:
def formatar_moeda(valor):
    """
    Formata um número float no padrão monetário brasileiro.
    Exemplo: 15000.5 -> R$ 15.000,50
    """
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

In [16]:
def exibir_relatorio(resumo_mensal, transacoes_validas, transacoes_invalidas, periodo):
    """
    Exibe o relatório financeiro mensal formatado no terminal.
    """

    print("===== RELATÓRIO FINANCEIRO CLEARBANK =====")
    print()
    print(f"Período analisado: {periodo['data_mais_antiga']} até {periodo['data_mais_recente']}")
    print(f"Dias analisados: {periodo['dias_analisados']}")
    print(f"Total de linhas lidas: {len(transacoes_validas) + len(transacoes_invalidas)}")
    print(f"Linhas válidas: {len(transacoes_validas)}")
    print(f"Linhas inválidas: {len(transacoes_invalidas)}")

    print()
    print("===== RELATÓRIO MENSAL =====")

    for mes in sorted(resumo_mensal.keys()):
        dados = resumo_mensal[mes]

        print()
        print(f"Mês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print()
    print("===== TRANSAÇÕES SUSPEITAS =====")

    transacoes_suspeitas = []

    for transacao in transacoes_validas:
        if transacao["suspeita"]:
            transacoes_suspeitas.append(transacao)

    if len(transacoes_suspeitas) == 0:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for transacao in transacoes_suspeitas:
            print(
                f"ID: {transacao['id']} | "
                f"Cliente: {transacao['cliente_id']} | "
                f"Data: {transacao['data_texto']} | "
                f"Valor: {formatar_moeda(transacao['valor'])}"
            )

In [18]:
def calcular_periodo(transacoes):
    """
    Calcula a data mais antiga, a data mais recente e a diferença em dias.
    """

    datas = [transacao["data"] for transacao in transacoes]

    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)
    dias_analisados = (data_mais_recente - data_mais_antiga).days

    return {
        "data_mais_antiga": data_mais_antiga.strftime("%Y-%m-%d"),
        "data_mais_recente": data_mais_recente.strftime("%Y-%m-%d"),
        "dias_analisados": dias_analisados
    }

In [20]:
def salvar_json(resumo_mensal, transacoes_validas, transacoes_invalidas, periodo, caminho_saida):
    """
    Salva o resultado da análise em um arquivo JSON.
    """

    transacoes_suspeitas = []

    for transacao in transacoes_validas:
        if transacao["suspeita"]:
            transacoes_suspeitas.append({
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data_texto"],
                "valor": transacao["valor"]
            })

    dados_json = {
        "gerado_em": date.today().strftime("%Y-%m-%d"),
        "periodo_analisado": periodo,
        "total_transacoes_validas": len(transacoes_validas),
        "total_transacoes_invalidas": len(transacoes_invalidas),
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": transacoes_suspeitas
    }

    try:
        with open(caminho_saida, "w", encoding="utf-8") as arquivo:
            json.dump(dados_json, arquivo, ensure_ascii=False, indent=2)

        print(f"Arquivo JSON gerado com sucesso: {caminho_saida}")

    except OSError as erro:
        print("Erro ao salvar o arquivo JSON.")
        print(erro)

In [21]:
salvar_json(
    resumo_mensal,
    transacoes_validas,
    transacoes_invalidas,
    periodo,
    CAMINHO_JSON
)

Arquivo JSON gerado com sucesso: relatorio.json


In [23]:
def executar_analise():
    """
    Executa o fluxo completo da análise financeira.
    """

    transacoes_brutas = ler_transacoes(CAMINHO_CSV)

    transacoes_validas = []
    transacoes_invalidas = []

    for linha in transacoes_brutas:
        transacao = validar_transacao(linha)

        if transacao is None:
            transacoes_invalidas.append(linha)
        else:
            transacoes_validas.append(transacao)

    if len(transacoes_validas) == 0:
        print("Nenhuma transação válida encontrada. A análise foi encerrada.")
        return

    resumo_mensal = gerar_relatorio(transacoes_validas)
    periodo = calcular_periodo(transacoes_validas)

    exibir_relatorio(
        resumo_mensal,
        transacoes_validas,
        transacoes_invalidas,
        periodo
    )

    salvar_json(
        resumo_mensal,
        transacoes_validas,
        transacoes_invalidas,
        periodo,
        CAMINHO_JSON
    )

    return resumo_mensal, transacoes_validas, transacoes_invalidas, periodo

In [24]:
resultado = executar_analise()

===== RELATÓRIO FINANCEIRO CLEARBANK =====

Período analisado: 2026-01-01 até 2026-06-30
Dias analisados: 180
Total de linhas lidas: 200
Linhas válidas: 185
Linhas inválidas: 15

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:    33
  Total crédito: R$ 78.195,29
  Total débito:  R$ 20.142,78
  Saldo:         R$ 58.052,51
  Média:         R$ 2.979,94
  Maior valor:   R$ 12.286,58
  Menor valor:   R$ 129,34

Mês: 2026-02
  Transações:    20
  Total crédito: R$ 65.410,71
  Total débito:  R$ 7.053,08
  Saldo:         R$ 58.357,63
  Média:         R$ 3.623,19
  Maior valor:   R$ 6.676,09
  Menor valor:   R$ 776,87

Mês: 2026-03
  Transações:    43
  Total crédito: R$ 83.947,58
  Total débito:  R$ 28.705,04
  Saldo:         R$ 55.242,54
  Média:         R$ 2.619,83
  Maior valor:   R$ 19.918,08
  Menor valor:   R$ 43,10

Mês: 2026-04
  Transações:    26
  Total crédito: R$ 49.739,22
  Total débito:  R$ 18.482,53
  Saldo:         R$ 31.256,69
  Média:         R$ 2.623,91
  Maior valo